In [ ]:
# =============================================================================
# SYSTEM CLEANUP (SSH) - RUN THIS FIRST
# =============================================================================
# This notebook generates LARGE EMBEDDINGS (~2-5 GB)
# This cell cleans up unnecessary files and checks disk space

import os
import sys
import gc
import shutil
import glob

def cleanup_before_embeddings(delete_old_embeddings=True, required_space_gb=5.0):
    """Clean up system before computing embeddings."""
    print("PRE-EMBEDDING CLEANUP")
    print("=" * 60)
    
    cleaned_total = 0
    
    # 1. Python garbage collection
    gc.collect()
    print("[OK] Python garbage collection")
    
    # 2. CUDA cache
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            print(f"[OK] CUDA cache cleared")
    except:
        pass
    
    # 3. Delete __pycache__
    for pycache in glob.glob('**/__pycache__', recursive=True):
        try:
            shutil.rmtree(pycache)
        except:
            pass
    print("[OK] __pycache__ deleted")
    
    # 4. Delete .ipynb_checkpoints
    for cp in glob.glob('**/.ipynb_checkpoints', recursive=True):
        try:
            shutil.rmtree(cp)
        except:
            pass
    print("[OK] .ipynb_checkpoints deleted")
    
    # 5. Delete old embeddings if requested
    emb_dir = './data/embeddings'
    if delete_old_embeddings and os.path.exists(emb_dir):
        print(f"\nDeleting old embeddings in {emb_dir}:")
        for f in os.listdir(emb_dir):
            if f.endswith('.npy'):
                fpath = os.path.join(emb_dir, f)
                size_mb = os.path.getsize(fpath) / 1e6
                try:
                    os.remove(fpath)
                    cleaned_total += size_mb
                    print(f"   [OK] {f} ({size_mb:.1f} MB)")
                except Exception as e:
                    print(f"   [ERROR] {f}: {e}")
    
    # 6. Show disk space
    total, used, free = shutil.disk_usage('/')
    free_gb = free / 1e9
    
    print(f"\nDisk space: {free_gb:.1f} GB free / {total/1e9:.1f} GB total")
    print(f"Cleaned: {cleaned_total:.1f} MB")
    
    if free_gb < required_space_gb:
        print(f"\nWARNING: Less than {required_space_gb} GB free!")
        print("   Embeddings may require 2-5 GB of space.")
        print("   Suggestion: Delete ~/.cache/huggingface if too large")
        
        # Show HF cache size
        hf_cache = os.path.expanduser('~/.cache/huggingface')
        if os.path.exists(hf_cache):
            hf_size = sum(os.path.getsize(os.path.join(dp, f)) 
                         for dp, dn, fn in os.walk(hf_cache) for f in fn)
            print(f"   HuggingFace Cache: {hf_size/1e9:.2f} GB")
    else:
        print(f"\n[OK] Sufficient space for embeddings!")
    
    return free_gb

# RUN CLEANUP
# Set delete_old_embeddings=False to keep old embeddings
free_space = cleanup_before_embeddings(
    delete_old_embeddings=True,  # WARNING: Deletes old embeddings!
    required_space_gb=5.0
)

print("\n" + "=" * 60)
print("[OK] Cleanup complete! You can run the notebook.")

In [ ]:
# ==============================================================================
# CONFIGURATION ET IMPORTS
# ==============================================================================

import json
import numpy as np
import pandas as pd
from pandas import json_normalize
import os
import re
import gc
import logging
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, RobustScaler
from sklearn.feature_selection import VarianceThreshold
import joblib

import torch
import torch.nn.functional as F
from transformers import CamembertTokenizer, CamembertModel
from tqdm.auto import tqdm

# Logging configuration
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ==============================================================================
# CONFIGURATION DATACLASS
# ==============================================================================

@dataclass
class PreprocessingConfig:
    """Configuration centralisée pour le preprocessing."""
    # Paths
    data_dir: str = "./data"
    embedding_dir: str = "./data/embeddings"
    feature_dir: str = "./data/features"
    
    # Columns
    text_col: str = "full_text"
    desc_col: str = "user.description"
    id_col: str = "challenge_id"
    
    # Model settings
    model_name: str = "camembert-base"
    max_length: int = 128
    batch_size: int = 32
    
    # Multi-layer embedding settings
    use_multi_layer: bool = True
    layers_to_use: Tuple[int, ...] = (-1, -2, -3, -4)  # Last 4 layers
    pooling_strategy: str = "attention"  # "cls", "mean", "attention"
    
    # Mixed precision
    use_fp16: bool = True
    
    def __post_init__(self):
        os.makedirs(self.data_dir, exist_ok=True)
        os.makedirs(self.embedding_dir, exist_ok=True)
        os.makedirs(self.feature_dir, exist_ok=True)

config = PreprocessingConfig()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Using device: {device}")

## 1. Data Loading

In [ ]:
# ==============================================================================
# DATA LOADING
# ==============================================================================

def load_jsonl_data(filepath: str) -> pd.DataFrame:
    """Load and normalize a JSONL file."""
    logger.info(f"Loading {filepath}...")
    df = pd.read_json(filepath, lines=True)
    df = json_normalize(df.to_dict(orient="records"))
    logger.info(f"Loaded {len(df)} records with {len(df.columns)} columns")
    return df

def extract_full_text(row: pd.Series) -> str:
    """Extract full text (extended if available)."""
    txt = row.get("text", "")
    extended = row.get("extended_tweet.full_text", np.nan)
    if pd.notna(extended) and extended:
        return str(extended)
    return str(txt) if pd.notna(txt) else ""

# Load data
train_df = load_jsonl_data("data/train.jsonl")
kaggle_df = load_jsonl_data("data/kaggle_test.jsonl")

# Separate features and labels
y_train = train_df["label"].values
X_train = train_df.drop("label", axis=1).copy()
X_kaggle = kaggle_df.copy()

# Extract full text
X_train[config.text_col] = X_train.apply(extract_full_text, axis=1)
X_kaggle[config.text_col] = X_kaggle.apply(extract_full_text, axis=1)

# Save labels
np.save(os.path.join(config.data_dir, "y_train.npy"), y_train)
logger.info(f"Labels distribution: {np.bincount(y_train)}")

print(f"\nData summary:")
print(f"   Train: {len(X_train)} samples")
print(f"   Test:  {len(X_kaggle)} samples")
print(f"   Labels: {np.sum(y_train == 0)} observers, {np.sum(y_train == 1)} influencers")

## 2. Feature Engineering - Text Features

In [ ]:
# ==============================================================================
# TEXTUAL FEATURE EXTRACTION
# ==============================================================================

try:
    import emoji
    HAS_EMOJI = True
except ImportError:
    HAS_EMOJI = False
    logger.warning("emoji package not installed, using fallback")

class TextFeatureExtractor:
    """Optimized textual feature extractor."""
    
    # Pre-compiled patterns for performance
    HASHTAG_PATTERN = re.compile(r'#\w+', re.UNICODE)
    MENTION_PATTERN = re.compile(r'@\w+', re.UNICODE)
    URL_PATTERN = re.compile(r'https?://\S+', re.UNICODE)
    RT_PATTERN = re.compile(r'(^RT @|\bRT\b|\bQT[:]?\b|RT\s@)', re.IGNORECASE)
    MEDIA_PATTERN = re.compile(r'photo|image|vid[eé]o|video|pic|gif', re.IGNORECASE)
    EMOJI_PATTERN = re.compile(r'[\U0001F300-\U0001F6FF\U0001F900-\U0001F9FF\u2600-\u27BF]')
    
    # Keywords for detection
    CTA_KEYWORDS = ['suivez', 'abonnez', "s'abonner", 'follow', 'subscribe', 
                    'retweet to win', 'like & follow', 'rt pour', 'rt if']
    PROMO_KEYWORDS = ['nouveau', 'article', 'video', 'lien', 'link', 
                      'disponible', 'promo', 'promotion', 'exclusif', 'decouvrez']
    INFLUENCER_KEYWORDS = ['partenariat', 'sponsorise', 'ad', 'pub', 'collab',
                           'concours', 'giveaway', 'code promo']
    
    @staticmethod
    def count_emojis(text: str) -> int:
        if not isinstance(text, str):
            return 0
        if HAS_EMOJI and hasattr(emoji, 'EMOJI_DATA'):
            return sum(1 for ch in text if ch in emoji.EMOJI_DATA)
        return len(TextFeatureExtractor.EMOJI_PATTERN.findall(text))
    
    @classmethod
    def extract_all(cls, texts: pd.Series) -> pd.DataFrame:
        """Extract all textual features."""
        texts = texts.astype(str)
        texts_lower = texts.str.lower()
        
        features = pd.DataFrame(index=texts.index)
        
        # Basic counts
        features['tweet_length'] = texts.str.len()
        features['word_count'] = texts.str.split().str.len().fillna(0)
        features['char_per_word'] = (features['tweet_length'] / features['word_count'].replace(0, 1)).fillna(0)
        
        # Hashtags
        features['hashtag_count'] = texts.str.count(cls.HASHTAG_PATTERN)
        features['is_hashtag_heavy'] = (features['hashtag_count'] > 3).astype(int)
        
        # Mentions
        features['mention_count'] = texts.str.count(cls.MENTION_PATTERN)
        features['is_mention_heavy'] = (features['mention_count'] > 2).astype(int)
        
        # URLs
        features['url_count'] = texts.str.count(cls.URL_PATTERN)
        features['has_url'] = (features['url_count'] > 0).astype(int)
        
        # Emojis
        features['emoji_count'] = texts.apply(cls.count_emojis)
        features['is_emoji_heavy'] = (features['emoji_count'] > 5).astype(int)
        
        # Punctuation
        features['exclamation_count'] = texts.str.count('!')
        features['question_count'] = texts.str.count('\\?')
        features['has_multiple_exclamations'] = (features['exclamation_count'] > 1).astype(int)
        
        # Uppercase ratio
        alpha_chars = texts.str.replace(r'[^a-zA-Z]', '', regex=True)
        upper_chars = texts.str.replace(r'[^A-Z]', '', regex=True)
        features['uppercase_ratio'] = (upper_chars.str.len() / alpha_chars.str.len().replace(0, 1)).fillna(0)
        
        # Content type detection
        features['has_rt_qt'] = texts.str.contains(cls.RT_PATTERN, na=False).astype(int)
        features['is_reply'] = texts.str.strip().str.startswith('@').astype(int)
        features['has_media_reference'] = texts.str.contains(cls.MEDIA_PATTERN, na=False).astype(int)
        
        # Keyword detection
        features['has_call_to_action'] = texts_lower.apply(
            lambda x: int(any(kw in x for kw in cls.CTA_KEYWORDS))
        )
        features['has_self_promotion'] = texts_lower.apply(
            lambda x: int(any(kw in x for kw in cls.PROMO_KEYWORDS))
        )
        features['has_influencer_keywords'] = texts_lower.apply(
            lambda x: int(any(kw in x for kw in cls.INFLUENCER_KEYWORDS))
        )
        
        # Engagement indicators
        features['engagement_score'] = (
            features['has_call_to_action'] * 2 +
            features['has_self_promotion'] +
            features['has_url'] +
            features['hashtag_count'].clip(upper=5) * 0.5
        )
        
        return features

# Extract text features
logger.info("Extracting text features...")
train_text_features = TextFeatureExtractor.extract_all(X_train[config.text_col])
kaggle_text_features = TextFeatureExtractor.extract_all(X_kaggle[config.text_col])

print(f"\nText features extracted: {train_text_features.shape[1]} features")
train_text_features.head()

## 3. Feature Engineering - Structured Features

In [ ]:
# ==============================================================================
# CLEAN AND SYNC COLUMNS
# ==============================================================================

def drop_complex_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Drop complex columns (nested objects)."""
    bad_cols = []
    for col in df.columns:
        if df[col].apply(lambda x: isinstance(x, (dict, list))).any():
            bad_cols.append(col)
    logger.info(f"Dropping {len(bad_cols)} complex columns")
    return df.drop(columns=bad_cols)

# Clean dataframes
X_train_clean = drop_complex_columns(X_train)
X_kaggle_clean = drop_complex_columns(X_kaggle)

# Synchronize columns
common_cols = list(set(X_train_clean.columns) & set(X_kaggle_clean.columns))
for col in [config.text_col, config.id_col]:
    if col in common_cols:
        common_cols.remove(col)

# Define column types
# NOTE: user.followers_count and user.friends_count DO NOT EXIST in the data!
# We only use actually available columns
NUMERIC_COLS = [
    # quoted_status.user features (these EXIST in quoted_status)
    "quoted_status.user.favourites_count", "quoted_status.favorite_count",
    "quoted_status.reply_count", "quoted_status.user.friends_count",
    "quoted_status.quote_count", "quoted_status.user.listed_count",
    "quoted_status.retweet_count", "quoted_status.user.followers_count",
    # user features (WITHOUT followers_count and friends_count - DO NOT EXIST!)
    "user.listed_count", "user.favourites_count", "user.statuses_count",
    # Tweet engagement
    "retweet_count", "favorite_count", "reply_count", "quote_count"
]
NUMERIC_COLS = [c for c in NUMERIC_COLS if c in common_cols]

CATEGORICAL_COLS = [
    "is_quote_status", "truncated", "possibly_sensitive",
    "user.geo_enabled", "user.is_translator", "user.default_profile",
    "user.profile_use_background_image", "user.translator_type"
]
CATEGORICAL_COLS = [c for c in CATEGORICAL_COLS if c in common_cols]

# ==============================================================================
# IMPROVED FEATURE ENGINEERING (based on feature_engineering.ipynb)
# ==============================================================================

def extract_source_device(source_str):
    """Extract device type from Twitter source.
    
    Very discriminative feature discovered in feature_engineering.ipynb!
    """
    if pd.isna(source_str):
        return 'unknown'
    source = str(source_str).lower()
    if 'iphone' in source:
        return 'iphone'
    elif 'android' in source:
        return 'android'
    elif 'tweetdeck' in source:
        return 'tweetdeck'
    elif 'web' in source or 'browser' in source:
        return 'web'
    elif any(x in source for x in ['buffer', 'hootsuite', 'socialflow', 'sprout', 'dlvr.it']):
        return 'bot_scheduler'
    else:
        return 'other'

def add_user_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add derived user features.
    
    IMPROVED with TOP features from feature_engineering.ipynb:
    - user_description_length (importance=736)
    - tweets_per_favourites (importance=699)
    - user_statuses_count (importance=639)
    - user_listed_count (importance=607)
    - listed_per_status (importance=553)
    - log_user_listed (corr=0.606)
    - total_engagement (very discriminative)
    - source_device (iphone/android/tweetdeck/bot)
    """
    df = df.copy()
    
    # === Base columns ===
    listed = df.get('user.listed_count', pd.Series([0]*len(df), index=df.index)).fillna(0)
    statuses = df.get('user.statuses_count', pd.Series([1]*len(df), index=df.index)).fillna(1).replace(0, 1)
    favourites = df.get('user.favourites_count', pd.Series([0]*len(df), index=df.index)).fillna(0)
    
    # === TOP Raw features (high LightGBM importance) ===
    df['user_statuses_count'] = statuses
    df['user_favourites_count'] = favourites
    df['user_listed_count'] = listed
    
    # === Highly discriminative ratios ===
    
    # Listed per status ratio (importance=553, Observer=0, Influencer=25 median)
    df['user_listed_per_status'] = listed / statuses
    df['user_listed_per_status'] = df['user_listed_per_status'].clip(upper=1)
    
    # Tweets per favourites ratio (importance=699!)
    df['user_tweets_per_favourites'] = statuses / (favourites + 1)
    df['user_tweets_per_favourites'] = df['user_tweets_per_favourites'].clip(upper=100)
    
    # === Log transforms (high correlation) ===
    df['log_user_statuses'] = np.log1p(statuses)      # corr=0.439
    df['log_user_favourites'] = np.log1p(favourites)
    df['log_user_listed'] = np.log1p(listed)           # corr=0.606
    
    # === Total Engagement (very discriminative composite feature) ===
    retweet = df.get('retweet_count', pd.Series([0]*len(df), index=df.index)).fillna(0)
    favorite = df.get('favorite_count', pd.Series([0]*len(df), index=df.index)).fillna(0)
    reply = df.get('reply_count', pd.Series([0]*len(df), index=df.index)).fillna(0)
    quote = df.get('quote_count', pd.Series([0]*len(df), index=df.index)).fillna(0)
    
    df['total_engagement'] = retweet + favorite + reply + quote
    df['log_total_engagement'] = np.log1p(df['total_engagement'])
    df['log_retweet_count'] = np.log1p(retweet)
    df['log_favorite_count'] = np.log1p(favorite)
    
    # === Source Device (very discriminative!) ===
    if 'source' in df.columns:
        df['source_device'] = df['source'].apply(extract_source_device)
    else:
        df['source_device'] = 'unknown'
    
    # === Highly discriminative binary features ===
    
    # Has banner (Observer=73%, Influencer=92%)
    if 'user.profile_banner_url' in df.columns:
        df['user_has_banner'] = df['user.profile_banner_url'].notna().astype(int)
    else:
        df['user_has_banner'] = 0
    
    # Has location (Observer=58%, Influencer=75%)
    if 'user.location' in df.columns:
        df['user_has_location'] = (df['user.location'].notna() & (df['user.location'] != '')).astype(int)
    else:
        df['user_has_location'] = 0
    
    # Has URL (Observer=16%, Influencer=56%)
    if 'user.url' in df.columns:
        df['user_has_url'] = df['user.url'].notna().astype(int)
    else:
        df['user_has_url'] = 0
    
    # Has description (importance=736!)
    if 'user.description' in df.columns:
        df['user_has_description'] = (df['user.description'].notna() & (df['user.description'] != '')).astype(int)
        df['user_desc_length'] = df['user.description'].fillna('').str.len()
        df['user_has_long_desc'] = (df['user_desc_length'] > 100).astype(int)
    else:
        df['user_has_description'] = 0
        df['user_desc_length'] = 0
        df['user_has_long_desc'] = 0
    
    # Default profile flags
    if 'user.default_profile' in df.columns:
        df['user_default_profile'] = df['user.default_profile'].fillna(False).astype(int)
    else:
        df['user_default_profile'] = 0
    
    if 'user.default_profile_image' in df.columns:
        df['user_default_profile_image'] = df['user.default_profile_image'].fillna(False).astype(int)
    else:
        df['user_default_profile_image'] = 0
    
    # === Is reply (very discriminative: Observer=39%, Influencer=19%) ===
    if 'in_reply_to_status_id' in df.columns:
        df['is_reply_real'] = df['in_reply_to_status_id'].notna().astype(int)
    else:
        df['is_reply_real'] = 0
    
    # Is quote status
    if 'is_quote_status' in df.columns:
        df['is_quote_status_flag'] = df['is_quote_status'].fillna(False).astype(int)
    else:
        df['is_quote_status_flag'] = 0
    
    # Has quoted status (tweet quotes another)
    if 'quoted_status.id' in df.columns:
        df['has_quoted_status'] = df['quoted_status.id'].notna().astype(int)
    else:
        df['has_quoted_status'] = 0
    
    return df

# Apply user features
X_train_clean = add_user_features(X_train_clean)
X_kaggle_clean = add_user_features(X_kaggle_clean)

# Update derived cols with ALL TOP features
DERIVED_COLS = [
    # TOP raw features (LightGBM importance)
    'user_statuses_count', 'user_favourites_count', 'user_listed_count',
    # Discriminative ratios
    'user_listed_per_status', 'user_tweets_per_favourites',
    # Log transforms (high correlation)
    'log_user_statuses', 'log_user_favourites', 'log_user_listed',
    # Engagement
    'total_engagement', 'log_total_engagement', 'log_retweet_count', 'log_favorite_count',
    # Binary features
    'user_has_banner', 'user_has_location', 'user_has_url',
    'user_has_description', 'user_desc_length', 'user_has_long_desc',
    'user_default_profile', 'user_default_profile_image',
    'is_reply_real', 'is_quote_status_flag', 'has_quoted_status'
]

# Source device will be one-hot encoded separately
SOURCE_DEVICE_COL = ['source_device']

print(f"\nStructured features (IMPROVED with feature_engineering.ipynb):")
print(f"   Numeric: {len(NUMERIC_COLS)} columns")
print(f"   Categorical: {len(CATEGORICAL_COLS)} columns")
print(f"   Derived: {len(DERIVED_COLS)} columns [TOP FEATURES]")
print(f"   Source device: one-hot encoding")

In [ ]:
# ==============================================================================
# SKLEARN PREPROCESSING PIPELINE (IMPROVED)
# ==============================================================================

# Combine all structured features
all_numeric_cols = NUMERIC_COLS + DERIVED_COLS

# Add source_device to categorical columns
all_categorical_cols = CATEGORICAL_COLS + SOURCE_DEVICE_COL

# Fill NaN values for numeric columns
for col in all_numeric_cols:
    if col in X_train_clean.columns:
        X_train_clean[col] = pd.to_numeric(X_train_clean[col], errors='coerce').fillna(0)
    if col in X_kaggle_clean.columns:
        X_kaggle_clean[col] = pd.to_numeric(X_kaggle_clean[col], errors='coerce').fillna(0)

# FIX: Convert categorical columns to strings (avoid mixed str/float from NaN)
for col in all_categorical_cols:
    if col in X_train_clean.columns:
        X_train_clean[col] = X_train_clean[col].fillna("missing").astype(str)
    if col in X_kaggle_clean.columns:
        X_kaggle_clean[col] = X_kaggle_clean[col].fillna("missing").astype(str)

print(f"Columns prepared (IMPROVED):")
print(f"   Numeric: {len(all_numeric_cols)} (includes TOP features from feature_engineering)")
print(f"   Categorical: {len(all_categorical_cols)} (includes source_device)")

# Create preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", RobustScaler())  # More robust to outliers
        ]), all_numeric_cols),
        ("cat", Pipeline([
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), all_categorical_cols),
    ],
    remainder="drop",
    n_jobs=-1
)

# Fit and transform
logger.info("Fitting preprocessing pipeline...")
X_train_struct = preprocessor.fit_transform(X_train_clean)
X_kaggle_struct = preprocessor.transform(X_kaggle_clean)

# Add text features
X_train_struct = np.hstack([X_train_struct, train_text_features.values])
X_kaggle_struct = np.hstack([X_kaggle_struct, kaggle_text_features.values])

# Save preprocessor
joblib.dump(preprocessor, os.path.join(config.feature_dir, "preprocessor.joblib"))

print(f"\n[OK] Structured features processed:")
print(f"   Train shape: {X_train_struct.shape}")
print(f"   Test shape: {X_kaggle_struct.shape}")
print(f"\nFeatures added from feature_engineering.ipynb:")
print(f"   - user_statuses_count, user_favourites_count, user_listed_count")
print(f"   - total_engagement, log_total_engagement")
print(f"   - source_device (one-hot: iphone/android/tweetdeck/bot/web/other)")
print(f"   - user_default_profile_image, has_quoted_status")

## 4. CamemBERT Embeddings (Multi-Layer)

In [ ]:
# ==============================================================================
# MULTI-LAYER CAMEMBERT EMBEDDINGS
# ==============================================================================

class CamemBERTEmbedder:
    """Multi-layer CamemBERT embedding extractor."""
    
    def __init__(self, config: PreprocessingConfig, device: torch.device):
        self.config = config
        self.device = device
        
        logger.info(f"Loading {config.model_name}...")
        self.tokenizer = CamembertTokenizer.from_pretrained(config.model_name)
        self.model = CamembertModel.from_pretrained(
            config.model_name,
            output_hidden_states=config.use_multi_layer
        )
        self.model.to(device)
        self.model.eval()
        
        # Attention weights for pooling
        if config.pooling_strategy == "attention":
            hidden_size = self.model.config.hidden_size
            self.attention = torch.nn.Linear(hidden_size, 1).to(device)
            torch.nn.init.xavier_uniform_(self.attention.weight)
        
        logger.info(f"Model loaded on {device}")
    
    def _pool_sequence(self, hidden_states: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        """Pool sequence according to chosen strategy."""
        if self.config.pooling_strategy == "cls":
            return hidden_states[:, 0, :]
        
        elif self.config.pooling_strategy == "mean":
            # Mean pooling with attention mask
            mask = attention_mask.unsqueeze(-1).expand(hidden_states.size()).float()
            sum_hidden = torch.sum(hidden_states * mask, dim=1)
            sum_mask = torch.clamp(mask.sum(dim=1), min=1e-9)
            return sum_hidden / sum_mask
        
        elif self.config.pooling_strategy == "attention":
            # Attention-weighted pooling
            mask = attention_mask.unsqueeze(-1).float()
            attn_scores = self.attention(hidden_states)  # (B, L, 1)
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
            attn_weights = F.softmax(attn_scores, dim=1)
            return torch.sum(hidden_states * attn_weights, dim=1)
        
        else:
            raise ValueError(f"Unknown pooling strategy: {self.config.pooling_strategy}")
    
    @torch.no_grad()
    def embed_texts(
        self, 
        texts: List[str], 
        show_progress: bool = True
    ) -> np.ndarray:
        """Generate embeddings for a list of texts."""
        embeddings = []
        batch_size = self.config.batch_size
        
        iterator = range(0, len(texts), batch_size)
        if show_progress:
            iterator = tqdm(iterator, desc="Embedding")
        
        for i in iterator:
            batch_texts = texts[i:i+batch_size]
            
            # Tokenize
            encoded = self.tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=self.config.max_length,
                return_tensors="pt"
            )
            input_ids = encoded['input_ids'].to(self.device)
            attention_mask = encoded['attention_mask'].to(self.device)
            
            # Forward pass with mixed precision
            if self.config.use_fp16 and self.device.type == "cuda":
                with torch.cuda.amp.autocast():
                    outputs = self.model(input_ids, attention_mask=attention_mask)
            else:
                outputs = self.model(input_ids, attention_mask=attention_mask)
            
            # Extract embeddings
            if self.config.use_multi_layer:
                # Concatenate multiple layers
                layer_embeddings = []
                for layer_idx in self.config.layers_to_use:
                    hidden = outputs.hidden_states[layer_idx]
                    pooled = self._pool_sequence(hidden, attention_mask)
                    layer_embeddings.append(pooled)
                batch_emb = torch.cat(layer_embeddings, dim=-1)
            else:
                batch_emb = self._pool_sequence(outputs.last_hidden_state, attention_mask)
            
            embeddings.append(batch_emb.cpu().numpy().astype(np.float32))
            
            # Memory cleanup
            if i % (batch_size * 50) == 0:
                gc.collect()
                if self.device.type == "cuda":
                    torch.cuda.empty_cache()
        
        return np.vstack(embeddings)
    
    def get_embedding_dim(self) -> int:
        """Return embedding dimension."""
        base_dim = self.model.config.hidden_size
        if self.config.use_multi_layer:
            return base_dim * len(self.config.layers_to_use)
        return base_dim

# Initialize embedder
embedder = CamemBERTEmbedder(config, device)
print(f"\nCamemBERT Embedder initialized")
print(f"   Embedding dimension: {embedder.get_embedding_dim()}")
print(f"   Pooling strategy: {config.pooling_strategy}")
print(f"   Multi-layer: {config.use_multi_layer} (layers: {config.layers_to_use})")

In [ ]:
# ==============================================================================
# GENERATE EMBEDDINGS
# WARNING: This cell can take a long time - Only run if necessary
# ==============================================================================

REGENERATE_EMBEDDINGS = True  # Set to True to regenerate

if REGENERATE_EMBEDDINGS:
    logger.info("Starting embedding generation...")
    
    # Prepare texts
    train_texts = X_train_clean[config.text_col].astype(str).tolist()
    train_desc = X_train_clean[config.desc_col].fillna("").astype(str).tolist()
    kaggle_texts = X_kaggle_clean[config.text_col].astype(str).tolist()
    kaggle_desc = X_kaggle_clean[config.desc_col].fillna("").astype(str).tolist()
    
    # Generate train text embeddings
    print("\nEmbedding TRAIN texts...")
    train_text_emb = embedder.embed_texts(train_texts)
    print(f"   Shape: {train_text_emb.shape}")
    
    # Generate train description embeddings
    print("\nEmbedding TRAIN descriptions...")
    train_desc_emb = embedder.embed_texts(train_desc)
    print(f"   Shape: {train_desc_emb.shape}")
    
    # Save SEPARATE embeddings for multi-branch architecture
    np.save(os.path.join(config.embedding_dir, "X_train_text_multilayer.npy"), train_text_emb)
    print(f"   [OK] Train TEXT embeddings saved: {train_text_emb.shape}")
    np.save(os.path.join(config.embedding_dir, "X_train_desc_multilayer.npy"), train_desc_emb)
    print(f"   [OK] Train DESC embeddings saved: {train_desc_emb.shape}")
    
    # Also save concatenated for backward compatibility
    train_full_emb = np.hstack([train_text_emb, train_desc_emb])
    np.save(os.path.join(config.embedding_dir, "X_train_multilayer_embeddings.npy"), train_full_emb)
    print(f"   [OK] Train COMBINED embeddings saved: {train_full_emb.shape}")
    del train_text_emb, train_desc_emb, train_full_emb
    gc.collect()
    
    # Generate kaggle text embeddings
    print("\nEmbedding KAGGLE texts...")
    kaggle_text_emb = embedder.embed_texts(kaggle_texts)
    print(f"   Shape: {kaggle_text_emb.shape}")
    
    # Generate kaggle description embeddings
    print("\nEmbedding KAGGLE descriptions...")
    kaggle_desc_emb = embedder.embed_texts(kaggle_desc)
    print(f"   Shape: {kaggle_desc_emb.shape}")
    
    # Save SEPARATE embeddings for multi-branch architecture
    np.save(os.path.join(config.embedding_dir, "X_kaggle_text_multilayer.npy"), kaggle_text_emb)
    print(f"   [OK] Kaggle TEXT embeddings saved: {kaggle_text_emb.shape}")
    np.save(os.path.join(config.embedding_dir, "X_kaggle_desc_multilayer.npy"), kaggle_desc_emb)
    print(f"   [OK] Kaggle DESC embeddings saved: {kaggle_desc_emb.shape}")
    
    # Also save concatenated for backward compatibility
    kaggle_full_emb = np.hstack([kaggle_text_emb, kaggle_desc_emb])
    np.save(os.path.join(config.embedding_dir, "X_kaggle_multilayer_embeddings.npy"), kaggle_full_emb)
    print(f"   [OK] Kaggle COMBINED embeddings saved: {kaggle_full_emb.shape}")
    del kaggle_text_emb, kaggle_desc_emb, kaggle_full_emb
    
    print(f"\n[OK] All embeddings saved (separate + combined)!")
    
    # Cleanup model and GPU memory
    del embedder
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    
    print("Memory cleaned up")
else:
    print("Skipping embedding generation (REGENERATE_EMBEDDINGS=False)")
    print("   Set REGENERATE_EMBEDDINGS=True to regenerate embeddings")

## 5. Final Assembly

In [ ]:
# ==============================================================================
# LOAD PREPROCESSED DATA (Skip all preprocessing)
# ==============================================================================

import numpy as np
import joblib
import gc
import os
from dataclasses import dataclass
from loguru import logger

@dataclass
class PreprocessingConfig:
    data_dir: str = "./data"
    embedding_dir: str = "./data/embeddings"
    feature_dir: str = "./data/features"

config = PreprocessingConfig()

print("Loading preprocessed files from disk...")

# Load structured features (from cell 14)
X_train_struct = np.load(os.path.join(config.feature_dir, "X_train_features.npy"))
X_kaggle_struct = np.load(os.path.join(config.feature_dir, "X_kaggle_features.npy"))
print(f"[OK] Structured features loaded: {X_train_struct.shape}, {X_kaggle_struct.shape}")

# Load embeddings (from cell 12)
X_train_emb = np.load(os.path.join(config.embedding_dir, "X_train_multilayer_embeddings.npy"))
X_kaggle_emb = np.load(os.path.join(config.embedding_dir, "X_kaggle_multilayer_embeddings.npy"))
print(f"[OK] Embeddings loaded: {X_train_emb.shape}, {X_kaggle_emb.shape}")

print("\nReady for final concatenation!")

In [ ]:
# ==============================================================================
# FINAL ASSEMBLY
# ==============================================================================

# Save structured features (FORCE overwrite to get new version)
train_feat_path = os.path.join(config.feature_dir, "X_train_features.npy")
kaggle_feat_path = os.path.join(config.feature_dir, "X_kaggle_features.npy")

np.save(train_feat_path, X_train_struct)
print(f"[OK] Saved: {train_feat_path}")

np.save(kaggle_feat_path, X_kaggle_struct)
print(f"[OK] Saved: {kaggle_feat_path}")

logger.info(f"Structured features: {X_train_struct.shape}")

# Clean up DataFrames - no longer needed
if 'X_train_clean' in dir():
    del X_train_clean
if 'X_kaggle_clean' in dir():
    del X_kaggle_clean
if 'train_text_features' in dir():
    del train_text_features
if 'kaggle_text_features' in dir():
    del kaggle_text_features
gc.collect()

# Load embeddings (use existing if not regenerated)
try:
    train_emb_path = os.path.join(config.embedding_dir, "X_train_multilayer_embeddings.npy")
    kaggle_emb_path = os.path.join(config.embedding_dir, "X_kaggle_multilayer_embeddings.npy")
    
    if os.path.exists(train_emb_path):
        X_train_emb = np.load(train_emb_path)
        X_kaggle_emb = np.load(kaggle_emb_path)
    else:
        # Fallback to single-layer embeddings
        X_train_emb = np.load(os.path.join(config.embedding_dir, "X_train_embeddings.npy"))
        X_kaggle_emb = np.load(os.path.join(config.embedding_dir, "X_kaggle_embeddings.npy"))
    
    logger.info(f"Embeddings loaded: {X_train_emb.shape}")
except FileNotFoundError as e:
    logger.error(f"Embedding files not found: {e}")
    logger.error("Please run the embedding generation cell first!")
    raise

In [ ]:
# ==============================================================================
# EXPORT FEATURES AS CSV (with proper column names)
# ==============================================================================

import numpy as np
import pandas as pd
import joblib
import os

# Load the saved preprocessor to get column names
preprocessor = joblib.load(os.path.join(config.feature_dir, "preprocessor.joblib"))

# Load the numpy feature arrays
X_train_struct = np.load(os.path.join(config.feature_dir, "X_train_features.npy"))
X_kaggle_struct = np.load(os.path.join(config.feature_dir, "X_kaggle_features.npy"))

print(f"Loaded structured features: {X_train_struct.shape}")

# Reconstruct column names from the preprocessor
numeric_cols = preprocessor.transformers_[0][2]  # numeric column names
categorical_cols = preprocessor.transformers_[1][2]  # categorical column names

# Get one-hot encoded categorical column names
onehot_encoder = preprocessor.transformers_[1][1].named_steps['onehot']
cat_feature_names = []
for i, cat_col in enumerate(categorical_cols):
    categories = onehot_encoder.categories_[i]
    cat_feature_names.extend([f"{cat_col}_{cat}" for cat in categories])

# Text feature names (22 features from TextFeatureExtractor)
text_feature_names = [
    'tweet_length', 'word_count', 'char_per_word',
    'hashtag_count', 'is_hashtag_heavy',
    'mention_count', 'is_mention_heavy',
    'url_count', 'has_url',
    'emoji_count', 'is_emoji_heavy',
    'exclamation_count', 'question_count', 'has_multiple_exclamations',
    'uppercase_ratio',
    'has_rt_qt', 'is_reply', 'has_media_reference',
    'has_call_to_action', 'has_self_promotion', 'has_influencer_keywords',
    'engagement_score'
]

# Combine all column names in order
all_column_names = list(numeric_cols) + cat_feature_names + text_feature_names

# Verify column count matches
if len(all_column_names) != X_train_struct.shape[1]:
    print(f"WARNING: Column mismatch: {len(all_column_names)} names vs {X_train_struct.shape[1]} features")
    print(f"   Using generic names for safety")
    all_column_names = [f'feature_{i}' for i in range(X_train_struct.shape[1])]
else:
    print(f"[OK] Column names reconstructed: {len(all_column_names)} features")

# Create DataFrames
train_feat_df = pd.DataFrame(X_train_struct, columns=all_column_names)
kaggle_feat_df = pd.DataFrame(X_kaggle_struct, columns=all_column_names)

# Save as CSV
train_csv_path = os.path.join(config.data_dir, "train_features.csv")
kaggle_csv_path = os.path.join(config.data_dir, "test_features.csv")

train_feat_df.to_csv(train_csv_path, index=False)
kaggle_feat_df.to_csv(kaggle_csv_path, index=False)

print(f"\n[OK] CSV files saved:")
print(f"   {train_csv_path} ({train_feat_df.shape})")
print(f"   {kaggle_csv_path} ({kaggle_feat_df.shape})")
print(f"\nSample column names:")
print(f"   Numeric: {all_column_names[:5]}")
print(f"   Categorical: {cat_feature_names[:5] if cat_feature_names else 'None'}")
print(f"   Text: {text_feature_names[:5]}")

# Show file sizes
import subprocess
result = subprocess.run(['du', '-h', train_csv_path, kaggle_csv_path], 
                       capture_output=True, text=True)
print(f"\nFile sizes:")
print(result.stdout)

In [ ]:
# ==============================================================================
# CONCATENATE ALL FEATURES
# ==============================================================================

# Final concatenation: structured features + embeddings
X_train_full = np.hstack([X_train_struct, X_train_emb]).astype(np.float32)
X_kaggle_full = np.hstack([X_kaggle_struct, X_kaggle_emb]).astype(np.float32)

# Free intermediate arrays
del X_train_struct, X_kaggle_struct, X_train_emb, X_kaggle_emb
gc.collect()

# Save final processed arrays (FORCE overwrite to get new version)
train_path = os.path.join(config.data_dir, "X_train_processed_multilayer.npy")
kaggle_path = os.path.join(config.data_dir, "X_kaggle_processed_multilayer.npy")

np.save(train_path, X_train_full)
print(f"[OK] Saved: {train_path}")

np.save(kaggle_path, X_kaggle_full)
print(f"[OK] Saved: {kaggle_path}")

# Calculate memory usage
train_size_gb = X_train_full.nbytes / (1024**3)
kaggle_size_gb = X_kaggle_full.nbytes / (1024**3)

print("\n" + "="*60)
print("PREPROCESSING COMPLETE")
print("="*60)
print(f"\nFinal shapes:")
print(f"   X_train: {X_train_full.shape} ({train_size_gb:.2f} GB)")
print(f"   X_kaggle: {X_kaggle_full.shape} ({kaggle_size_gb:.2f} GB)")
print(f"\nFiles saved to:")
print(f"   {config.data_dir}/X_train_processed_multilayer.npy")
print(f"   {config.data_dir}/X_kaggle_processed_multilayer.npy")
print(f"   {config.feature_dir}/preprocessor.joblib")

In [ ]:
# ==============================================================================
# VALIDATION & MEMORY CLEANUP
# ==============================================================================

def validate_data(X: np.ndarray, name: str) -> bool:
    """Validate processed data."""
    issues = []
    
    # Check for NaN
    nan_count = np.isnan(X).sum()
    if nan_count > 0:
        issues.append(f"Found {nan_count} NaN values")
    
    # Check for Inf
    inf_count = np.isinf(X).sum()
    if inf_count > 0:
        issues.append(f"Found {inf_count} Inf values")
    
    # Check for very large values
    large_count = (np.abs(X) > 1e6).sum()
    if large_count > 0:
        issues.append(f"Found {large_count} values > 1e6")
    
    if issues:
        print(f"WARNING {name}: {', '.join(issues)}")
        return False
    else:
        print(f"[OK] {name}: All checks passed")
        return True

print("\nValidating processed data...")
validate_data(X_train_full, "X_train")
validate_data(X_kaggle_full, "X_kaggle")

# Quick stats
print(f"\nData statistics:")
print(f"   Mean: {X_train_full.mean():.4f}")
print(f"   Std:  {X_train_full.std():.4f}")
print(f"   Min:  {X_train_full.min():.4f}")
print(f"   Max:  {X_train_full.max():.4f}")

# Final memory cleanup - free everything except final arrays
print("\nFinal memory cleanup...")
del X_train_full, X_kaggle_full
gc.collect()

# Show disk usage
import subprocess
result = subprocess.run(['du', '-sh', config.data_dir], capture_output=True, text=True)
print(f"\nDisk usage for {config.data_dir}: {result.stdout.strip()}")